In [1]:
# Librerias necesarias
import pandas as pd

In [10]:
df = pd.read_csv('03-classification/course_lead_scoring_2026.csv')
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [12]:
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
num_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

for col in cat_cols:
    df[col] = df[col].fillna('NA')
for col in num_cols:
    df[col] = df[col].fillna(0)

### Question 1

What is the most frequent observation (mode) for the column industry?
- NA
- technology
- healthcare
- retail


In [13]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

In [ ]:
df[num_cols].corr()
# interaction_count and lead_score

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


### Split the data

In [23]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


### Question 3

Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only. Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?
- industry
- location
- lead_source
- employment_status


In [24]:
from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, df_train['converted'])

mi = df_train[cat_cols].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

### Question 4

Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?
- 0.55
- 0.65
- 0.75
- 0.85


In [28]:
from sklearn.feature_extraction import DictVectorizer

cols = cat_cols + num_cols
cols.remove('converted')

dv = DictVectorizer(sparse=False)
train_dicts = df_train[cols].to_dict(orient='records')
val_dicts = df_val[cols].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
y_train = df_train['converted']
X_val = dv.transform(val_dicts)
y_val = df_val['converted']

In [29]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred_val = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred_val >= 0.5)

In [33]:
from sklearn.metrics import accuracy_score

ref_value = accuracy_score(y_val, converted_decision.astype(int))
print(ref_value)

0.645


### Question 5

Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'

Note: The difference doesn't have to be positive.


In [ ]:
remove_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
for col in remove_candidates:
    cols = cat_cols + num_cols
    cols.remove('converted')
    cols.remove(col)

    dv = DictVectorizer(sparse=False)
    train_dicts = df_train[cols].to_dict(orient='records')
    val_dicts = df_val[cols].to_dict(orient='records')

    X_train = dv.fit_transform(train_dicts)
    y_train = df_train['converted']
    X_val = dv.transform(val_dicts)
    y_val = df_val['converted']

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred_val = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred_val >= 0.5)
    print(f'Model without {col}')
    acc = accuracy_score(y_val, converted_decision.astype(int))
    print(f'Model accuracy: {acc}')
    print(f'Diff vs original: {ref_value - acc}')

# number_of_courses_viewed

Model without lead_source
Model accuracy: 0.642
Diff vs original: 0.0030000000000000027
Model without number_of_courses_viewed
Model accuracy: 0.643
Diff vs original: 0.0020000000000000018
Model without interaction_count
Model accuracy: 0.601
Diff vs original: 0.04400000000000004


### Question 6

Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001


In [ ]:
C_list = [0.000001, 0.00001, 0.0001, 0.001]

for c in C_list:
    cols = cat_cols + num_cols
    cols.remove('converted')

    dv = DictVectorizer(sparse=False)
    train_dicts = df_train[cols].to_dict(orient='records')
    val_dicts = df_val[cols].to_dict(orient='records')

    X_train = dv.fit_transform(train_dicts)
    y_train = df_train['converted']
    X_val = dv.transform(val_dicts)
    y_val = df_val['converted']

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred_val = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred_val >= 0.5)
    print(f'Model C: {c:.6f}')
    acc = round(accuracy_score(y_val, converted_decision.astype(int)), 3)
    print(f'Model accuracy: {acc}')

# 0.000001

Model C: 0.000001
Model accuracy: 0.598
Model C: 0.000010
Model accuracy: 0.598
Model C: 0.000100
Model accuracy: 0.613
Model C: 0.001000
Model accuracy: 0.645
